In [1]:
from oggm import cfg, utils
from oggm import workflow
from oggm import DEFAULT_BASE_URL
from oggm import GlacierDirectory

import os
import geopandas as gpd
import xarray as xr
import xvec
from oggmzarr.datacube.geozarr import OggmZarrHandler, GeoZarrHandler

In [2]:
cfg.initialize(logging_level="ERROR")

cfg.PARAMS["use_multiprocessing"] = True
cfg.PARAMS["continue_on_error"] = True
cfg.PATHS['working_dir'] = utils.gettempdir(dirname='oggm-geozarr', reset=True)
rgi_ids = ['RGI60-11.00897', "RGI60-06.00377"] # HEF, Bruarjoekull 

DEFAULT_BASE_URL

2026-04-16 15:25:46: oggm.cfg: Reading default parameters from the OGGM `params.cfg` configuration file.
2026-04-16 15:25:46: oggm.cfg: Multiprocessing switched OFF according to the parameter file.
2026-04-16 15:25:46: oggm.cfg: Multiprocessing: using all available processors (N=22)
2026-04-16 15:25:46: oggm.cfg: Multiprocessing switched ON after user settings.
2026-04-16 15:25:46: oggm.cfg: PARAMS['continue_on_error'] changed from `False` to `True`.


'https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L3-L5_files/2025.6/elev_bands/W5E5/per_glacier_spinup/'

# Level 4

We start with L4 as this is known to be DTCG-compatible, so we can reuse the GeoZarrHandler out-of-the-box

In [3]:
gdirs = workflow.init_glacier_directories(
    rgi_ids,
    prepro_base_url=DEFAULT_BASE_URL,  # where to fetch the data?
    from_prepro_level=4,  # what kind of data? 
    prepro_border=80  # how big of a map?
)


2026-04-16 15:25:47: oggm.workflow: init_glacier_directories from prepro level 4 on 2 glaciers.
2026-04-16 15:25:47: oggm.workflow: Execute entity tasks [gdir_from_prepro] on 2 glaciers


In [4]:
gdir = gdirs[0]
gdir

<oggm.GlacierDirectory>
  RGI id: RGI60-11.00897
  Region: 11: Central Europe
  Subregion: 11-01: Alps                            
  Name: Hintereisferner
  Glacier type: Glacier
  Terminus type: Land-terminating
  Status: Glacier or ice cap
  Area: 8.036 km2
  Lon, Lat: (10.7584, 46.8003)
  Grid (nx, ny): (278, 236)
  Grid (dx, dy): (50.0, -50.0)

In [5]:
gdir.__dict__.keys()

dict_keys(['extent_ll', 'rgi_id', 'glims_id', 'cenlon', 'cenlat', 'rgi_region', 'rgi_subregion', 'rgi_version', 'rgi_dem_source', 'glacier_type', 'terminus_type', 'status', 'name', 'rgi_region_name', 'rgi_subregion_name', 'is_tidewater', 'is_lake_terminating', 'is_marine_terminating', 'is_shelf_terminating', 'is_nominal', 'inversion_calving_rate', 'is_icecap', 'hemisphere', 'rgi_date', 'base_dir', 'dir', 'logfile', '_mbdf', '_mbprofdf', '_mbprofdf_cte_dh', '_lazy_rgi_area_km2', '_lazy_grid'])

In [6]:
def get_oggm_datacube(gdir):
        with xr.open_dataset(gdir.get_filepath("gridded_data")) as ds:
            ds = ds.load()
            keywords = (
                "extent_l1"
                'terminus',
                'status',
                'is_',
                'inversion_calving_rate', 
                'hemisphere',
                'base_dir',
                'dir',
                'logfile',
                "rgi",
                "glacier",
                "name",
                "terminus",
                "cenl",  # latitude, longitude
                "glims",
            )
            glacier_attributes = {
                key: val
                for key, val in gdir.__dict__.items()
                if any(k in key for k in keywords)
            }
            print(glacier_attributes)
            # glacier_attributes = dict(gdir.__dict__.items())
            ds.attrs.update({"glacier_attributes": glacier_attributes})

        return ds
def add_rgi_id_to_dataset(dataset, gdir):
        """

        Parameters
        ----------
        dataset

        Returns
        -------

        """
        dataset.attrs["RGI-ID"] = gdir.rgi_id
        return dataset

In [7]:
datacube_l1 = get_oggm_datacube(gdir=gdir)
datacube_l1 = add_rgi_id_to_dataset(dataset=datacube_l1, gdir=gdir)

{'rgi_id': 'RGI60-11.00897', 'glims_id': 'G010758E46800N', 'cenlon': 10.7584, 'cenlat': 46.8003, 'rgi_region': '11', 'rgi_subregion': '11-01', 'rgi_version': '60', 'rgi_dem_source': 'COPDEM30', 'glacier_type': 'Glacier', 'terminus_type': 'Land-terminating', 'status': 'Glacier or ice cap', 'name': 'Hintereisferner', 'rgi_region_name': '11: Central Europe', 'rgi_subregion_name': '11-01: Alps                            ', 'is_tidewater': False, 'is_lake_terminating': False, 'is_marine_terminating': False, 'is_shelf_terminating': False, 'is_nominal': False, 'inversion_calving_rate': 0.0, 'is_icecap': False, 'hemisphere': 'nh', 'rgi_date': 2003, 'base_dir': '/tmp/OGGM/oggm-geozarr/per_glacier', 'dir': '/tmp/OGGM/oggm-geozarr/per_glacier/RGI60-11/RGI60-11.00/RGI60-11.00897', 'logfile': '/tmp/OGGM/oggm-geozarr/per_glacier/RGI60-11/RGI60-11.00/RGI60-11.00897/log.txt', '_lazy_rgi_area_km2': 8.036}


In [17]:
gdir.dir

'/tmp/OGGM/oggm-geozarr/per_glacier/RGI60-11/RGI60-11.00/RGI60-11.00897'

In [8]:
datacube_handler = GeoZarrHandler(ds=datacube_l1)

datacube_handler.data_tree

<xarray.DataTree>
Group: /
└── Group: /L1
        Dimensions:          (y: 236, x: 278)
        Coordinates:
          * y                (y) float32 944B 5.191e+06 5.191e+06 ... 5.179e+06
          * x                (x) float32 1kB 6.277e+05 6.277e+05 ... 6.415e+05 6.415e+05
            spatial_ref      int64 8B 0
        Data variables:
            topo             (y, x) float32 262kB 2.467e+03 2.464e+03 ... 2.281e+03
            topo_smoothed    (y, x) float32 262kB 2.461e+03 2.465e+03 ... 2.279e+03
            topo_valid_mask  (y, x) int8 66kB 1 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1 1
            glacier_mask     (y, x) int8 66kB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
            glacier_ext      (y, x) int8 66kB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
        Attributes:
            Conventions:         CF-1.12
            comment:             The DTC Glaciers project is developed under the Euro...
            date_created:        2026-04-16T15:25:47.931584
            RGI-ID:              RGI60-11.00897
            glacier_attributes:  {'rgi_id': 'RGI60-11.00897', 'glims_id': 'G010758E46...
            title:               Datacube of glacier-domain variables.
            summary:             Resampled glacier-domain variables from multiple sou...

## Minimum files for L4

In [9]:
print(os.listdir(gdir.dir))

['elevation_band_flowline.csv', 'model_geometry_spinup_historical.nc', 'inversion_flowlines.pkl', 'dem_source.txt', 'climate_historical.nc', 'model_diagnostics_spinup_historical.nc', 'inversion_input.pkl', 'gridded_data.nc', 'fl_diagnostics_historical.nc', 'model_geometry_historical.nc', 'downstream_line.pkl', 'model_flowlines.pkl', 'glacier_grid.json', 'diagnostics.json', 'mb_calib.json', 'log.txt', 'intersects.tar.gz', 'inversion_output.pkl', 'fl_diagnostics_spinup_historical.nc', 'model_diagnostics_historical.nc', 'outlines.tar.gz', 'model_flowlines_dyn_melt_f_calib.pkl', 'dem.tif']


## Adding netCDF to zarr

This requires adding metadata mapping to `metadata_mapping_data.yaml`.
One possible roadblock are identical variables names for different time resolutions.

In [10]:
# with xr.open_dataset(gdir.get_filepath("gridded_data")) as ds:
#     ds = ds.load()
#     ds.attrs["RGI-ID"] = gdir.rgi_id
#     datacube_handler.add_layer(ds=ds, ds_name="gridded_data", overwrite=True)

In [11]:
with xr.open_dataset(gdir.get_filepath("climate_historical")) as ds:
    ds = ds.load()
    ds.attrs["RGI-ID"] = gdir.rgi_id
    ds = ds.assign_coords(coords = {"rgi_id":gdir.rgi_id})
    ds = ds.expand_dims(("rgi_id")).swap_dims()
    # datacube_handler.add_layer(ds=ds, ds_name="climate_historical",overwrite=True)
    datacube_handler.add_datacube(datacubes={"monthly":ds}, datacube_name="climate_historical")

In [12]:
datacube_handler.data_tree

<xarray.DataTree>
Group: /
├── Group: /L1
│       Dimensions:          (y: 236, x: 278)
│       Coordinates:
│         * y                (y) float32 944B 5.191e+06 5.191e+06 ... 5.179e+06
│         * x                (x) float32 1kB 6.277e+05 6.277e+05 ... 6.415e+05 6.415e+05
│           spatial_ref      int64 8B 0
│       Data variables:
│           topo             (y, x) float32 262kB 2.467e+03 2.464e+03 ... 2.281e+03
│           topo_smoothed    (y, x) float32 262kB 2.461e+03 2.465e+03 ... 2.279e+03
│           topo_valid_mask  (y, x) int8 66kB 1 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1 1
│           glacier_mask     (y, x) int8 66kB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
│           glacier_ext      (y, x) int8 66kB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
│       Attributes:
│           Conventions:         CF-1.12
│           comment:             The DTC Glaciers project is developed under the Euro...
│           date_created:        2026-04-16T15:25:47.931584
│           RGI-ID:              RGI60-11.00897
│           glacier_attributes:  {'rgi_id': 'RGI60-11.00897', 'glims_id': 'G010758E46...
│           title:               Datacube of glacier-domain variables.
│           summary:             Resampled glacier-domain variables from multiple sou...
└── Group: /L2_climate_historical
    └── Group: /L2_climate_historical/monthly
            Dimensions:   (rgi_id: 1, time: 1428)
            Coordinates:
              * rgi_id    (rgi_id) <U14 56B 'RGI60-11.00897'
              * time      (time) datetime64[ns] 11kB 1901-01-01 1901-02-01 ... 2019-12-01
            Data variables:
                prcp      (rgi_id, time) float32 6kB 17.59 29.17 120.8 ... 67.95 170.7 52.07
                temp      (rgi_id, time) float32 6kB -9.867 -12.65 -7.506 ... 3.3 -3.0 -4.1
                temp_std  (rgi_id, time) float32 6kB 4.757 5.564 3.448 ... 2.344 3.095 2.942
            Attributes:
                Conventions:         CF-1.12
                comment:             The DTC Glaciers project is developed under the Euro...
                date_created:        2026-04-16T15:25:47.982190
                RGI-ID:              RGI60-11.00897
                glacier_attributes:  {}
                title:               Datacube of observation-informed modelled variables.
                summary:             Observation-informed modelled variables for RGI6-ID ...

# Storing shapefiles

In [20]:
datacube_handler.data_tree["outlines"] = gdir.read_shapefile("outlines").to_xarray()
xvec_shapefile = datacube_handler.get_layer("outlines").xvec.to_geodataframe()
assert gdir.read_shapefile("outlines").equals(xvec_shapefile)
datacube_handler.data_tree.outlines

/tmp/ipykernel_87428/3926142745.py:2: UserWarning: No active geometry column to be set. The resulting object will be a pandas.DataFrame with geopandas.GeometryArray(s) containing geometry and CRS information. Use `.set_geometry()` to set an active geometry and upcast to the geopandas.GeoDataFrame manually.
  xvec_shapefile = datacube_handler.get_layer("outlines").xvec.to_geodataframe()


<xarray.DataTree 'outlines'>
Group: /outlines
    Dimensions:     (index: 1)
    Coordinates:
      * index       (index) int64 8B 0
    Data variables: (12/25)
        RGIId       (index) object 8B 'RGI60-11.00897'
        GLIMSId     (index) object 8B 'G010758E46800N'
        BgnDate     (index) object 8B '20030799'
        EndDate     (index) object 8B '20030999'
        CenLon      (index) object 8B '10.7584'
        CenLat      (index) object 8B '46.8003'
        ...          ...
        Surging     (index) object 8B '9'
        Linkages    (index) object 8B '1'
        Name        (index) object 8B 'Hintereisferner'
        check_geom  (index) object 8B None
        dem_source  (index) object 8B 'COPDEM30'
        geometry    (index) geometry 8B POLYGON ((633569.9999925077 5185961.99999...

# Export via the DTCG pipeline

In [14]:
datacube_handler.data_tree.L2_climate_historical.monthly.time.attrs

{'standard_name': 'time', 'long_name': 'N/A', 'units': 'Float year'}

In [15]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory(suffix=".zarr") as tmpdir:
    output_path = Path(f"{gdir.rgi_id}.zarr")
    datacube_handler.export(output_path)
    print(f"✅ GeoZarr exported to: {output_path}")
    items = sorted(p.name for p in output_path.iterdir())
    print("📂 Top-level contents:", ", ".join(items) or "(empty)")

ValueError: Key 'units' already exists in attrs, and will not be overwritten. This is probably an encoding field used by xarray to describe how a variable is serialized. To proceed, remove this key from the variable's attributes manually.

# Let's try streaming!

In [ ]:
stream_url = "https://cluster.klima.uni-bremen.de/~dtcg/test_oggm_zarr/RGI60-11.00897.zarr/"

In [ ]:
stream_cube = xr.open_datatree(
    stream_url,
    group="",
    chunks={},
    engine="zarr",
    consolidated=True,
    decode_cf=True,
)
stream_cube.gridded_data

In [ ]:
stream_cube = xr.open_datatree(
    stream_url,
    group="",
    chunks={},
    engine="zarr",
    consolidated=True,
    decode_cf=True,
)
stream_cube